# Kiểm thử model YOLO11 (đã huấn luyện) cho phát hiện tàu thuyền trên ảnh Sentinel-2

**Bối cảnh.** Notebook này *không huấn luyện lại* mà chỉ **kiểm thử (inference + đánh giá)** một model
YOLO11 đã được huấn luyện sẵn trong bài báo:

> Mäyrä J. và cộng sự (2025), *Mapping recreational marine traffic from Sentinel-2 imagery using YOLO
> object detection models*, Remote Sensing of Environment 326:114791.

- **Model:** `yolo11s_tci.pt` — biến thể `yolo11s` fine-tune trên dữ liệu **L1C-TCI** (True Colour Image),
  là model đạt điểm cao nhất trong bài báo (test F1 = 0.863, mAP50 = 0.888 *sau hậu xử lý GIS*).
  Checkpoint lấy từ HuggingFace [`mayrajeo/marine-vessel-yolo`](https://huggingface.co/mayrajeo/marine-vessel-yolo/tree/main).
- **Code tham chiếu:** [github.com/mayrajeo/ship-detection](https://github.com/mayrajeo/ship-detection)
  (dùng `ultralytics` + định dạng nhãn YOLO, `single_cls=True`, lớp duy nhất là *vessel/boat*).
- **Dữ liệu test:** cùng bộ test đang dùng (ảnh **800×800 px**), nhưng đánh giá theo **định dạng nhãn YOLOv11**.
  Notebook tự chuyển đổi COCO JSON → YOLO `.txt` nếu cần.

**Đầu ra:**
1. Chỉ số độ chính xác chuẩn của object detection: **Precision, Recall, F1, mAP50, mAP50-95** + số **TP/FP/FN**.
2. **Visualise** các ảnh test kèm bounding box: ground-truth (xanh) và dự đoán của model (đỏ).

> ⚠️ **Lưu ý về so sánh với bài báo.** Con số F1=0.863 trong bài báo là *sau* bước hậu xử lý GIS
> (loại bỏ các phát hiện trên đất liền, đá ngầm, trại cá… bằng các lớp bản đồ của Phần Lan). Notebook này
> đánh giá **độ chính xác thô của bộ phát hiện** trên các chip 800×800, nên kết quả gần với *Bảng 4*
> (trước hậu xử lý) chứ không phải con số 0.863. Ngoài ra đây là bộ test cụ thể của bạn, không phải hai tile
> gốc 34VEN/34VER của bài báo.

**Môi trường:** notebook này độc lập với môi trường InternImage/mmdet của các notebook khác trong repo.
Nó chỉ cần `ultralytics` (như repo của mayrajeo) chạy trên Kaggle GPU — xem Section 2.

## Section 0 — Cấu hình (SỬA CHỖ NÀY TRƯỚC KHI CHẠY)

Bật **Settings → Accelerator → GPU** trong Kaggle. Trỏ `TEST_IMAGES_DIR` / `TEST_COCO_JSON` về bộ test của bạn
(mặc định dùng chính dataset `annotated-sentinel/test` như các notebook InternImage).

In [ ]:
import os

CFG = dict(
    # ===== Checkpoint YOLO11 đã huấn luyện (HuggingFace mayrajeo/marine-vessel-yolo) =====
    HF_REPO_ID   = "mayrajeo/marine-vessel-yolo",
    HF_FILENAME  = "yolo11s_tci.pt",
    # Nếu đã tải sẵn file .pt vào một Kaggle Dataset, trỏ đường dẫn ở đây để BỎ QUA bước tải từ HF
    # (hữu ích khi notebook không có Internet). Ví dụ: "/kaggle/input/yolo11s-tci/yolo11s_tci.pt"
    LOCAL_WEIGHTS = "",

    # ===== Bộ dữ liệu test =====
    # Hỗ trợ 2 định dạng đầu vào:
    #   "coco" : thư mục ảnh + 1 file COCO JSON (giống dataset annotated-sentinel hiện có) -> tự chuyển sang YOLO
    #   "yolo" : đã có sẵn images/ + labels/ theo chuẩn YOLO (mỗi ảnh 1 file .txt cùng tên)
    TEST_FORMAT      = "coco",
    TEST_IMAGES_DIR  = "/kaggle/input/datasets/hunglq/annotated-sentinel/test",
    TEST_COCO_JSON   = "/kaggle/input/datasets/hunglq/annotated-sentinel/test/_annotations.coco.json",
    # (chỉ dùng khi TEST_FORMAT="yolo") thư mục labels YOLO; để trống nếu labels nằm ở <images>/../labels
    TEST_YOLO_LABELS = "",

    # ===== Tham số suy luận / đánh giá =====
    IMG_SIZE   = 800,     # ảnh test là 800x800 -> giữ nguyên độ phân giải (800 chia hết cho 32).
                          # Tàu rất nhỏ (4-18px): có thể thử 1280/1600 để tăng recall, đổi lại chậm hơn.
    CONF_EVAL  = 0.001,   # ngưỡng thấp khi val để dựng đúng đường cong PR & tính mAP
    CONF_VIS   = 0.25,    # ngưỡng confidence khi VẼ dự đoán (bài báo dùng ~0.286 cho model này)
    IOU_NMS    = 0.7,     # IoU cho NMS
    CLASS_NAME = "vessel",

    # ===== Thư mục làm việc (đều nằm trong /kaggle/working để giữ lại sau khi chạy) =====
    WORK_ROOT  = "/kaggle/working",
    YOLO_DIR   = "/kaggle/working/yolo_test",     # nơi build images/ + labels/ + data.yaml
    OUT_DIR    = "/kaggle/working/eval_outputs",  # nơi lưu metric + hình visualise
    N_VIS      = 12,      # số ảnh test đem ra visualise
    SEED       = 42,
)

os.makedirs(CFG["YOLO_DIR"], exist_ok=True)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)

# Kiểm tra nhanh đường dẫn dữ liệu
assert os.path.isdir(CFG["TEST_IMAGES_DIR"]), f"Không thấy thư mục ảnh test: {CFG['TEST_IMAGES_DIR']}"
if CFG["TEST_FORMAT"] == "coco":
    assert os.path.isfile(CFG["TEST_COCO_JSON"]), f"Không thấy COCO JSON: {CFG['TEST_COCO_JSON']}"
print("CFG OK — test format:", CFG["TEST_FORMAT"])
print("  images:", CFG["TEST_IMAGES_DIR"])

## Section 1 — Kiểm tra GPU & môi trường

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Section 2 — Cài đặt môi trường YOLOv11

Chỉ cần gói `ultralytics` (giống repo của mayrajeo). Bài báo dùng `ultralytics==8.3.74`, PyTorch 2.6, Python 3.12.
Trên Kaggle đã có sẵn PyTorch, nên chỉ cài thêm `ultralytics`. `huggingface_hub` dùng để tải checkpoint.

Nếu bản pin gây xung đột version trên image Kaggle, có thể đổi thành `pip install -q -U ultralytics`
(bản mới vẫn hỗ trợ YOLO11).

In [ ]:
%%capture install_log
!pip install -q "ultralytics==8.3.74" "huggingface_hub>=0.24"

In [ ]:
import ultralytics
ultralytics.checks()
print("ultralytics version:", ultralytics.__version__)

## Section 3 — Tải checkpoint `yolo11s_tci.pt`

Ưu tiên dùng `LOCAL_WEIGHTS` nếu đã trỏ; nếu không thì tải từ HuggingFace. Nếu Kaggle notebook đang tắt Internet,
hãy add file `.pt` như một Dataset rồi điền `LOCAL_WEIGHTS` ở Section 0.

In [ ]:
from pathlib import Path

def resolve_weights(cfg):
    if cfg["LOCAL_WEIGHTS"] and os.path.exists(cfg["LOCAL_WEIGHTS"]):
        print("Dùng weights local:", cfg["LOCAL_WEIGHTS"])
        return cfg["LOCAL_WEIGHTS"]
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(
        repo_id=cfg["HF_REPO_ID"],
        filename=cfg["HF_FILENAME"],
        local_dir=os.path.join(cfg["WORK_ROOT"], "weights"),
    )
    print("Đã tải checkpoint:", p)
    return p

WEIGHTS = resolve_weights(CFG)
assert os.path.exists(WEIGHTS)

## Section 4 — Chuẩn bị dữ liệu test theo định dạng YOLOv11

YOLO/ultralytics cần cấu trúc:

```
YOLO_DIR/
├── images/   *.png|*.jpg|*.tif   (ảnh test 800×800)
├── labels/   *.txt               (mỗi dòng: <cls> <xc> <yc> <w> <h>, toạ độ chuẩn hoá 0..1)
└── data.yaml (path, val, test, names)
```

- Nếu `TEST_FORMAT="coco"`: chuyển box COCO `[x,y,w,h]` (pixel, góc trên-trái) → YOLO `[xc,yc,w,h]` (chuẩn hoá).
  Mọi annotation gộp về **1 lớp duy nhất** (class 0), khớp với model huấn luyện `single_cls`.
- Nếu `TEST_FORMAT="yolo"`: chỉ copy ảnh + labels có sẵn.

In [ ]:
import json, glob, shutil
from pathlib import Path
from PIL import Image

IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

def _reset_dir(d):
    d = Path(d)
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    return d

def build_from_coco(cfg):
    out_img = _reset_dir(Path(cfg["YOLO_DIR"]) / "images")
    out_lbl = _reset_dir(Path(cfg["YOLO_DIR"]) / "labels")
    img_dir = cfg["TEST_IMAGES_DIR"]
    with open(cfg["TEST_COCO_JSON"]) as f:
        coco = json.load(f)
    images = {im["id"]: im for im in coco["images"]}
    anns_by_img = {}
    for a in coco.get("annotations", []):
        anns_by_img.setdefault(a["image_id"], []).append(a)

    n_img, n_box = 0, 0
    for iid, im in images.items():
        fn = im["file_name"]
        src = os.path.join(img_dir, fn)
        if not os.path.exists(src):
            continue
        W, H = im.get("width"), im.get("height")
        if not W or not H:
            with Image.open(src) as _im:
                W, H = _im.size
        dst_img = out_img / Path(fn).name
        shutil.copy(src, dst_img)
        lines = []
        for a in anns_by_img.get(iid, []):
            x, y, w, h = a["bbox"]                 # COCO: pixel, góc trên-trái
            xc = min(max((x + w / 2) / W, 0.0), 1.0)
            yc = min(max((y + h / 2) / H, 0.0), 1.0)
            ww = min(max(w / W, 0.0), 1.0)
            hh = min(max(h / H, 0.0), 1.0)
            if ww <= 0 or hh <= 0:
                continue
            lines.append(f"0 {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}")
            n_box += 1
        (out_lbl / (Path(fn).stem + ".txt")).write_text("\n".join(lines))
        n_img += 1
    print(f"[COCO→YOLO] {n_img} ảnh, {n_box} box. -> {cfg['YOLO_DIR']}")
    return n_img, n_box

def build_from_yolo(cfg):
    out_img = _reset_dir(Path(cfg["YOLO_DIR"]) / "images")
    out_lbl = _reset_dir(Path(cfg["YOLO_DIR"]) / "labels")
    img_dir = Path(cfg["TEST_IMAGES_DIR"])
    lbl_dir = Path(cfg["TEST_YOLO_LABELS"]) if cfg["TEST_YOLO_LABELS"] else img_dir.parent / "labels"
    assert lbl_dir.is_dir(), f"Không thấy thư mục labels YOLO: {lbl_dir}"
    imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    n_img, n_box = 0, 0
    for src in imgs:
        shutil.copy(src, out_img / src.name)
        src_lbl = lbl_dir / (src.stem + ".txt")
        if src_lbl.exists():
            txt = src_lbl.read_text()
            n_box += sum(1 for ln in txt.splitlines() if ln.strip())
            # ép mọi class về 0 (single-cls)
            fixed = []
            for ln in txt.splitlines():
                p = ln.split()
                if len(p) >= 5:
                    fixed.append("0 " + " ".join(p[1:5]))
            (out_lbl / (src.stem + ".txt")).write_text("\n".join(fixed))
        else:
            (out_lbl / (src.stem + ".txt")).write_text("")
        n_img += 1
    print(f"[YOLO→YOLO] {n_img} ảnh, {n_box} box. -> {cfg['YOLO_DIR']}")
    return n_img, n_box

if CFG["TEST_FORMAT"] == "coco":
    n_img, n_box = build_from_coco(CFG)
else:
    n_img, n_box = build_from_yolo(CFG)
assert n_img > 0, "Không có ảnh test nào được tạo — kiểm tra lại đường dẫn."

In [ ]:
# Ghi data.yaml cho ultralytics
import yaml

DATA_YAML = os.path.join(CFG["YOLO_DIR"], "data.yaml")
data_cfg = {
    "path": CFG["YOLO_DIR"],
    "train": "images",   # không dùng để train, chỉ để yaml hợp lệ
    "val":   "images",
    "test":  "images",
    "names": {0: CFG["CLASS_NAME"]},
}
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)
print(open(DATA_YAML).read())

## Section 5 — Đánh giá độ chính xác (Precision / Recall / F1 / mAP)

Dùng `model.val()` của ultralytics với `single_cls=True` (bỏ qua sai lệch tên lớp giữa model và data.yaml).
`conf` để thấp (0.001) để dựng đầy đủ đường cong PR, từ đó ultralytics báo P/R tại điểm F1 cao nhất, cùng mAP50 và mAP50-95.

In [ ]:
from ultralytics import YOLO

model = YOLO(WEIGHTS)
print("Tên lớp trong model:", model.names)

metrics = model.val(
    data=DATA_YAML,
    split="val",
    imgsz=CFG["IMG_SIZE"],
    conf=CFG["CONF_EVAL"],
    iou=CFG["IOU_NMS"],
    single_cls=True,
    plots=True,                 # xuất confusion matrix, PR curve, F1 curve...
    save_json=False,
    project=CFG["OUT_DIR"],
    name="val",
    exist_ok=True,
    verbose=True,
)

In [ ]:
import numpy as np, json as _json

# --- Metric tổng hợp ---
P    = float(metrics.box.mp)      # precision trung bình (tại điểm F1 tốt nhất)
R    = float(metrics.box.mr)      # recall trung bình
mAP50    = float(metrics.box.map50)
mAP5095  = float(metrics.box.map)
F1   = 2 * P * R / (P + R + 1e-9)

# --- TP / FP / FN từ confusion matrix (single-class -> ma trận 2x2) ---
cm = metrics.confusion_matrix.matrix   # hàng = dự đoán, cột = thực tế; index 0=vessel, 1=background
TP = int(cm[0, 0]); FP = int(cm[0, 1]); FN = int(cm[1, 0])

# --- Ngưỡng confidence cho F1 cao nhất (giống cách bài báo chọn ngưỡng) ---
try:
    f1c = metrics.box.f1_curve
    px  = metrics.box.px           # trục confidence 0..1
    f1_mean = f1c.mean(0) if f1c.ndim > 1 else f1c
    best_conf = float(px[int(np.argmax(f1_mean))])
except Exception:
    best_conf = None

results = {
    "num_test_images": int(n_img),
    "num_gt_boxes":    int(n_box),
    "imgsz":           CFG["IMG_SIZE"],
    "precision":       round(P, 4),
    "recall":          round(R, 4),
    "f1":              round(F1, 4),
    "mAP50":           round(mAP50, 4),
    "mAP50_95":        round(mAP5095, 4),
    "TP": TP, "FP": FP, "FN": FN,
    "best_f1_conf":    None if best_conf is None else round(best_conf, 4),
}
with open(os.path.join(CFG["OUT_DIR"], "metrics.json"), "w") as f:
    _json.dump(results, f, indent=2)

print("================ KẾT QUẢ ĐÁNH GIÁ (YOLO11s_tci trên test 800×800) ================")
for k, v in results.items():
    print(f"  {k:16s}: {v}")
print("Đã lưu:", os.path.join(CFG["OUT_DIR"], "metrics.json"))

In [ ]:
# Bảng gọn + hiển thị các biểu đồ ultralytics đã xuất (PR curve, F1 curve, confusion matrix)
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

display(pd.DataFrame([{
    "Precision": results["precision"], "Recall": results["recall"], "F1": results["f1"],
    "mAP50": results["mAP50"], "mAP50-95": results["mAP50_95"],
    "TP": results["TP"], "FP": results["FP"], "FN": results["FN"],
}]))

val_dir = os.path.join(CFG["OUT_DIR"], "val")
for name in ["confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    fp = os.path.join(val_dir, name)
    if os.path.exists(fp):
        plt.figure(figsize=(6, 5))
        plt.imshow(mpimg.imread(fp)); plt.axis("off"); plt.title(name)
        plt.show()

## Section 6 — Visualise ảnh test kèm nhãn

Với mỗi ảnh: vẽ **ground-truth (xanh lá)** và **dự đoán của model (đỏ, kèm confidence)** ở ngưỡng `CONF_VIS`.
Tiêu đề mỗi ảnh ghi số GT và số dự đoán để dễ đối chiếu.

In [ ]:
import random, glob
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def read_yolo_gt(label_path, W, H):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    for ln in open(label_path):
        p = ln.split()
        if len(p) < 5:
            continue
        xc, yc, w, h = map(float, p[1:5])
        x1 = (xc - w / 2) * W; y1 = (yc - h / 2) * H
        boxes.append((x1, y1, w * W, h * H))   # (x, y, w, h) pixel
    return boxes

img_paths = sorted(glob.glob(os.path.join(CFG["YOLO_DIR"], "images", "*")))
random.seed(CFG["SEED"]); random.shuffle(img_paths)
sel = img_paths[:CFG["N_VIS"]]

preds = model.predict(sel, imgsz=CFG["IMG_SIZE"], conf=CFG["CONF_VIS"],
                      iou=CFG["IOU_NMS"], verbose=False)

ncol = 3
nrow = (len(sel) + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 6 * nrow))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

vis_dir = os.path.join(CFG["OUT_DIR"], "visualisations")
os.makedirs(vis_dir, exist_ok=True)

for ax, img_path, res in zip(axes, sel, preds):
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    ax.imshow(im)
    # Ground truth (xanh lá)
    lbl = os.path.join(CFG["YOLO_DIR"], "labels", os.path.splitext(os.path.basename(img_path))[0] + ".txt")
    gts = read_yolo_gt(lbl, W, H)
    for (x, y, w, h) in gts:
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="lime", linewidth=1.5))
    # Dự đoán (đỏ)
    n_pred = 0
    if res.boxes is not None and len(res.boxes) > 0:
        xyxy = res.boxes.xyxy.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        n_pred = len(xyxy)
        for (x1, y1, x2, y2), c in zip(xyxy, confs):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="red", linewidth=1.2))
            ax.text(x1, max(y1 - 2, 0), f"{c:.2f}", color="red", fontsize=7,
                    bbox=dict(facecolor="white", alpha=0.6, pad=0, edgecolor="none"))
    ax.set_title(f"{os.path.basename(img_path)}\nGT={len(gts)}  Pred={n_pred}", fontsize=9)
    ax.axis("off")

for ax in axes[len(sel):]:
    ax.axis("off")

# chú thích màu
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([0], [0], color="lime", lw=2, label="Ground truth"),
                    Line2D([0], [0], color="red",  lw=2, label=f"Prediction (conf≥{CFG['CONF_VIS']})")],
           loc="upper center", ncol=2, fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.98])
out_fig = os.path.join(vis_dir, "test_predictions_grid.png")
plt.savefig(out_fig, dpi=120, bbox_inches="tight")
print("Đã lưu:", out_fig)
plt.show()

### (Tuỳ chọn) Lưu từng ảnh dự đoán riêng lẻ bằng bộ vẽ sẵn của ultralytics

In [ ]:
# Xuất ảnh có sẵn box dự đoán (chỉ prediction) do ultralytics tự vẽ
per_img_dir = os.path.join(CFG["OUT_DIR"], "pred_per_image")
os.makedirs(per_img_dir, exist_ok=True)
for img_path, res in zip(sel, preds):
    out = os.path.join(per_img_dir, os.path.basename(img_path))
    res.save(filename=out)
print(f"Đã lưu {len(sel)} ảnh dự đoán vào: {per_img_dir}")

## Section 7 — Ghi chú & khắc phục sự cố

- **Metric thấp hơn 0.863 của bài báo là bình thường.** Bài báo đạt F1=0.863 *sau* hậu xử lý GIS
  (loại phát hiện trên đất/đá/trại cá bằng các lớp bản đồ Phần Lan) và trên toàn tile 34VEN/34VER.
  Đây là đánh giá thô trên chip 800×800, gần với *Bảng 4* (trước hậu xử lý).
- **Tàu quá nhỏ (4–18 px):** nếu recall thấp, thử tăng `IMG_SIZE` (1280 hoặc 1600) ở Section 0 — model gốc
  được huấn luyện trên chip 320px *phóng to 2× lên 640px* nên nó "quen" nhìn vật thể lớn hơn native.
- **Chọn ngưỡng confidence:** `best_f1_conf` in ở Section 5 là ngưỡng cho F1 cao nhất; đặt `CONF_VIS` bằng
  giá trị đó để phần visualise khớp với điểm đánh giá (bài báo dùng ~0.286 cho `yolo11s_tci`).
- **Không có Internet trên Kaggle:** add file `yolo11s_tci.pt` như một Dataset rồi điền `LOCAL_WEIGHTS`.
- **Xung đột version ultralytics/numpy:** đổi Section 2 thành `pip install -q -U ultralytics`.
- **Bộ test đã sẵn định dạng YOLO:** đặt `TEST_FORMAT="yolo"` và trỏ `TEST_IMAGES_DIR` (+ `TEST_YOLO_LABELS`).

Toàn bộ kết quả (metrics.json, biểu đồ, ảnh visualise) nằm trong `/kaggle/working/eval_outputs/`.